# TP Jour 3 — Spark SQL Avancé, Rigueur Statistique & Couche Gold

### Objectifs de la séance
1. Diagnostiquer l'**asymétrie (`SKEWNESS`)** de vos variables quantitatives et justifier l'emploi d'indicateurs robustes (**Médiane `PERCENTILE_APPROX`**, **IQR `P75 - P25`**, **P90**) plutôt que la seule moyenne arithmétique.
2. Écrire des requêtes **Spark SQL avancées** mobilisant des CTE (`WITH ...`) et des **fonctions de fenêtrage** (`DENSE_RANK`, `ROW_NUMBER`, cumul de **Pareto**, variation temporelle **`LAG`**).
3. Construire au moins **2 à 3 tables agrégées métier dans `/lakehouse/gold/`** prêtes à alimenter votre Dashboard Streamlit à faible latence.
4. Éviter le *Small Files Problem* sur HDFS et compacter votre table Silver avec `OPTIMIZE`.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("J3_Analytics_SQL_Gold")
    .getOrCreate()
)

# Table Silver du Jour 2 (grain = 1 avis enrichi du catalogue) exposée en vue SQL `v_silver`
SILVER_MAIN_PATH = "/lakehouse/silver/steam_reviews_enriched"
spark.read.format("delta").load(SILVER_MAIN_PATH).createOrReplaceTempView("v_silver")
print("Lignes disponibles dans v_silver :", spark.sql("SELECT COUNT(*) FROM v_silver").collect()[0][0])

# Vue `v_games` (grain = 1 jeu). Prix, temps de jeu et propriétaires sont des attributs du catalogue,
# répétés sur chacun des avis du jeu : les étudier au grain avis pondérerait chaque jeu par son nombre
# d'avis. Les distributions se calculent donc au grain jeu ; volume et taux de recommandation viennent
# des avis. Petite table (quelques milliers de lignes) mise en cache car réutilisée par toutes les requêtes.
spark.sql("""
CREATE OR REPLACE TEMP VIEW v_games AS
SELECT app_id,
       ANY_VALUE(game_name)                    AS game_name,
       ANY_VALUE(primary_genre)                AS primary_genre,
       ANY_VALUE(publisher)                    AS publisher,
       ANY_VALUE(release_date)                 AS release_date,
       ANY_VALUE(release_year)                 AS release_year,
       ANY_VALUE(price)                        AS price,
       ANY_VALUE(price_tier)                   AS price_tier,
       ANY_VALUE(average_playtime)             AS average_playtime,
       ANY_VALUE(owners_mid)                   AS owners_mid,
       COUNT(*)                                AS n_reviews,
       AVG(CAST(is_recommended AS INT))        AS recommendation_rate,
       AVG(CAST(is_helpful AS INT))            AS helpful_rate
FROM v_silver
GROUP BY app_id
""")
spark.sql("CACHE TABLE v_games")
n_games = spark.sql("SELECT COUNT(*) FROM v_games").collect()[0][0]
print("Jeux distincts dans v_games  :", n_games)
assert n_games > 0, "v_games est vide : relancez les Jours 1 et 2."

# Catalogue complet (dimension Silver du Jour 2, y compris les jeux sans aucun avis) : dénominateur de la
# couverture du dataset d'avis par année de sortie (Gold #2)
spark.read.format("delta").load("/lakehouse/silver/steam_games").createOrReplaceTempView("v_catalog")

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a07cb630-7bc5-4fc9-84d8-439b60d90e7d;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found

Lignes disponibles dans v_silver : 3763740


Jeux distincts dans v_games  : 8935


## 1. Analyse d'Asymétrie : Moyenne vs Médiane & Écart Interquartile (IQR)

### Exercice 1.1 — Étudier la distribution de votre métrique quantitative principale
Écrivez une requête Spark SQL sur `v_silver` calculant sur votre variable continue principale :
- `COUNT(*)`, `AVG(...)`, `STDDEV(...)`, `SKEWNESS(...)`
- Les quantiles `P25`, `P50` (médiane), `P75`, `P90`, `P95` via `PERCENTILE_APPROX(col, q)` et `MAX(...)`.

Rédigez ensuite en Markdown **votre interprétation statistique** (écart moyenne/médiane, impact des valeurs extrêmes, choix de l'indicateur retenu pour votre dashboard).

In [2]:
# Résumé statistique au grain jeu des 4 variables quantitatives de la problématique.
# Format long (UNION ALL) : 1 ligne par variable, mêmes indicateurs pour toutes.
df_dist_summary = spark.sql("""
WITH long_metrics AS (
    SELECT 'n_reviews' AS metric, CAST(n_reviews AS DOUBLE) AS value FROM v_games
    UNION ALL SELECT 'price', price FROM v_games
    UNION ALL SELECT 'average_playtime', CAST(average_playtime AS DOUBLE) FROM v_games
    UNION ALL SELECT 'owners_mid', CAST(owners_mid AS DOUBLE) FROM v_games
)
SELECT metric,
       COUNT(*)                        AS n_games,
       ROUND(AVG(value), 2)            AS mean,
       ROUND(STDDEV(value), 2)         AS stddev,
       ROUND(SKEWNESS(value), 2)       AS skewness,
       PERCENTILE_APPROX(value, 0.25)  AS p25,
       PERCENTILE_APPROX(value, 0.50)  AS p50_median,
       PERCENTILE_APPROX(value, 0.75)  AS p75,
       PERCENTILE_APPROX(value, 0.90)  AS p90,
       PERCENTILE_APPROX(value, 0.95)  AS p95,
       MAX(value)                      AS max,
       PERCENTILE_APPROX(value, 0.75) - PERCENTILE_APPROX(value, 0.25)    AS iqr,
       ROUND(try_divide(AVG(value), PERCENTILE_APPROX(value, 0.50)), 2)   AS mean_to_median
FROM long_metrics
GROUP BY metric
ORDER BY skewness DESC
""")
assert df_dist_summary is not None, "TODO : Écrivez la requête SQL de résumé statistique."
df_dist_summary.show(truncate=False)

# Métrique principale = volume d'avis par jeu : poids des valeurs extrêmes (règle de Tukey, > P75 + 1,5 x IQR)
dist = {r["metric"]: r for r in df_dist_summary.collect()}
main = dist["n_reviews"]
tukey_fence = main["p75"] + 1.5 * main["iqr"]
outliers = spark.sql(f"""
SELECT COUNT(*) AS n_outliers, SUM(n_reviews) AS outlier_reviews, (SELECT SUM(n_reviews) FROM v_games) AS all_reviews
FROM v_games WHERE n_reviews > {tukey_fence}
""").first()
print(f"Seuil de Tukey (P75 + 1,5 x IQR) : {tukey_fence:,.0f} avis -> {outliers['n_outliers']:,} jeux atypiques "
      f"({outliers['n_outliers'] / main['n_games']:.1%} des jeux) portant "
      f"{outliers['outlier_reviews'] / outliers['all_reviews']:.1%} des avis")
assert main["skewness"] > 1, "Asymétrie à droite attendue sur le volume d'avis par jeu"
assert main["mean"] > main["p50_median"], "Distribution étirée à droite : moyenne > médiane"

+----------------+-------+---------+----------+--------+-------+----------+--------+--------+---------+--------+--------+--------------+
|metric          |n_games|mean     |stddev    |skewness|p25    |p50_median|p75     |p90     |p95      |max     |iqr     |mean_to_median|
+----------------+-------+---------+----------+--------+-------+----------+--------+--------+---------+--------+--------+--------------+
|owners_mid      |8935   |314132.06|1988397.22|50.78   |10000.0|35000.0   |150000.0|350000.0|1500000.0|1.5E8   |140000.0|8.98          |
|average_playtime|8935   |317.24   |2943.41   |41.34   |0.0    |0.0       |208.0   |418.0   |838.0    |190625.0|208.0   |NULL          |
|n_reviews       |8935   |421.24   |2064.39   |14.58   |12.0   |37.0      |155.0   |661.0   |1675.0   |57182.0 |143.0   |11.38         |
|price           |8935   |6.85     |7.01      |4.45    |2.79   |4.99      |8.59    |14.99   |18.99    |154.99  |5.8     |1.37          |
+----------------+-------+---------+-----

Seuil de Tukey (P75 + 1,5 x IQR) : 370 avis -> 1,310 jeux atypiques (14.7% des jeux) portant 87.8% des avis


**Votre justification statistique *(À compléter)* :**
> **Grain d'analyse.** Prix, temps de jeu et propriétaires sont des attributs du *jeu*, répétés sur chacun de ses avis dans Silver. Calculés au grain avis, ils pondéreraient chaque jeu par sa popularité (un jeu à 50 000 avis compterait 50 000 fois). Les distributions sont donc étudiées sur la vue `v_games` (**8 935 jeux ayant au moins un avis**, 1 ligne par jeu, construite par `GROUP BY app_id`). *Mesures du 2026-11-18.*
>
> | Variable (grain jeu) | Moyenne | Médiane | Moyenne / médiane | Écart-type | Skewness | IQR (P75 - P25) | P95 | Max |
> |---|---|---|---|---|---|---|---|---|
> | `n_reviews` (avis par jeu, **métrique principale**) | 421 | **37** | ×11,4 | 2 064 | **14,6** | 143 (12 → 155) | 1 675 | 57 182 |
> | `price` (£) | 6,85 | **4,99** | ×1,4 | 7,01 | 4,5 | 5,80 (2,79 → 8,59) | 18,99 | 154,99 |
> | `average_playtime` (min) | 317 | **0** | indéfini | 2 943 | 41,3 | 208 (0 → 208) | 838 | 190 625 |
> | `owners_mid` (propriétaires) | 314 132 | **35 000** | ×9,0 | 1 988 397 | 50,8 | 140 000 | 1 500 000 | 150 000 000 |
>
> - **Toutes les variables sont fortement asymétriques à droite.** Le coefficient d'asymétrie de Fisher (`SKEWNESS`) dépasse partout le seuil usuel de |1| : de 4,5 pour le prix à 50,8 pour les propriétaires. Une loi normale aurait un skewness proche de 0. Dans chaque cas l'écart-type dépasse (ou égale) la moyenne, signe d'une queue longue.
> - **Volume d'avis par jeu (métrique principale).** La moyenne (421 avis) est **11 fois** la médiane (37 avis). Elle est tirée vers le haut par quelques blockbusters (max 57 182 avis), alors que le jeu « typique » en reçoit quelques dizaines. Avec la règle de Tukey (au-delà de P75 + 1,5 × IQR = 370 avis), **1 310 jeux (14,7 %) sont atypiques et portent 87,8 % des avis**. Les retirer comme « outliers » supprimerait donc l'essentiel du marché : ce sont des valeurs extrêmes **légitimes**, propres à un marché de type « winner takes all », pas des erreurs.
> - **`average_playtime` est surtout faite de zéros** : la médiane vaut 0, donc plus de la moitié des jeux n'ont pas de temps de jeu mesuré par SteamSpy. La moyenne (317 min) ne décrit aucun jeu réel. Pour le Jour 4, cette variable demandera un traitement à part (indicateur « temps de jeu mesuré » + `log1p` sur les valeurs positives).
> - **`owners_mid` est une variable discrète** (milieu de tranches `"20000-50000"`…) : ses quantiles tombent sur des milieux de tranche. Elle reste exploitable en ordre de grandeur, d'où la transformation logarithmique prévue au Jour 4.
>
> **Indicateurs retenus pour le dashboard.** La moyenne a un point de rupture de 0 % : une seule valeur extrême suffit à la déplacer sans limite. La **médiane** et l'**IQR** ont un point de rupture de 50 % et de 25 % : ils décrivent le jeu typique et la dispersion de la moitié centrale du marché sans être perturbés par la queue. Les tables Gold affichent donc **médiane + IQR** pour les comparaisons entre groupes. Les **sommes** (volume total d'avis) servent pour les parts de marché et le Pareto, où les extrêmes sont justement l'information recherchée. Les taux de recommandation sont des proportions bornées dans [0, 1] et se comparent directement.
>
> `PERCENTILE_APPROX` est un quantile **approché** (algorithme Greenwald-Khanna, précision par défaut 10 000). L'erreur de rang est inférieure à n / 10 000, donc moins d'un jeu ici : c'est négligeable, et cela évite le tri complet qu'exigerait un quantile exact sur de gros volumes.

---
## 2. Table Gold #1 : Classement Métier & Loi de Pareto (`OVER ORDER BY ...`)

### Exercice 2.1 — Agrégation par segment/catégorie + Courbe cumulée de Pareto
Construisez en Spark SQL une première table Gold agrégée par une dimension métier pertinente de votre dataset incluant :
- Le volume d'observations, le total et les indicateurs robustes (**médiane**, **IQR**).
- Le rang via `DENSE_RANK() OVER (ORDER BY ... DESC)`.
- Le pourcentage cumulé du total (**analyse de Pareto 80/20**) via :
  `SUM(metric_total) OVER (ORDER BY metric_total DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / SUM(metric_total) OVER ()`.

Sauvegardez-la avec `.coalesce(1)` au format **Delta** dans `/lakehouse/gold/<nom_gold_1>`.

In [3]:
GOLD_PARETO_PATH = "/lakehouse/gold/genre_pareto"

# Gold #1 : 1 ligne par genre principal. Volume et satisfaction (part d'avis positifs) + indicateurs robustes
# au grain jeu (médiane, IQR), rang DENSE_RANK et cumul de Pareto sur le volume d'avis.
df_gold_1 = spark.sql("""
WITH by_genre AS (
    SELECT primary_genre,
           COUNT(*)                                                    AS n_games,
           SUM(n_reviews)                                              AS total_reviews,
           ROUND(SUM(n_reviews * recommendation_rate) / SUM(n_reviews), 4) AS recommendation_rate,
           ROUND(AVG(n_reviews), 1)                                    AS mean_reviews_per_game,
           PERCENTILE_APPROX(n_reviews, 0.5)                           AS median_reviews_per_game,
           PERCENTILE_APPROX(n_reviews, 0.75) - PERCENTILE_APPROX(n_reviews, 0.25) AS iqr_reviews_per_game,
           ROUND(PERCENTILE_APPROX(recommendation_rate, 0.5), 4)       AS median_game_recommendation_rate,
           PERCENTILE_APPROX(price, 0.5)                               AS median_price,
           ROUND(PERCENTILE_APPROX(price, 0.75) - PERCENTILE_APPROX(price, 0.25), 2) AS iqr_price
    FROM v_games
    GROUP BY primary_genre
),
pareto AS (
    SELECT *,
           DENSE_RANK() OVER (ORDER BY total_reviews DESC)             AS rank_by_reviews,
           ROUND(total_reviews / SUM(total_reviews) OVER (), 4)        AS share_reviews,
           ROUND(SUM(total_reviews) OVER (ORDER BY total_reviews DESC, primary_genre
                                          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
                 / SUM(total_reviews) OVER (), 4)                      AS pareto_cum_share
    FROM by_genre
)
SELECT *,
       CASE WHEN pareto_cum_share - share_reviews < 0.80 THEN 'A'
            WHEN pareto_cum_share - share_reviews < 0.95 THEN 'B'
            ELSE 'C' END                                               AS pareto_class
FROM pareto
ORDER BY rank_by_reviews, primary_genre
""")
assert df_gold_1 is not None, "TODO : Écrivez la requête SQL avec fenêtre de Pareto dans `df_gold_1`."

# Petite table agrégée -> 1 seul fichier (coalesce(1)) : pas de small files côté Gold
(
    df_gold_1.coalesce(1)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_PARETO_PATH)
)
df_gold_1.show(15, truncate=False)

gold_1 = spark.read.format("delta").load(GOLD_PARETO_PATH)
last_cum = gold_1.agg({"pareto_cum_share": "max"}).first()[0]
assert abs(last_cum - 1.0) < 1e-3, "Le cumul de Pareto doit atteindre 100 %"
n_a = gold_1.filter("pareto_class = 'A'").count()
print(f"{n_a} genre(s) sur {gold_1.count()} (classe A) concentrent ~80 % des avis.")

# Pareto au grain jeu : quelle part des jeux concentre 80 % des avis ?
concentration = spark.sql("""
WITH ranked AS (
    SELECT n_reviews,
           ROW_NUMBER() OVER (ORDER BY n_reviews DESC, app_id) AS rn,
           COUNT(*) OVER ()                                    AS n,
           SUM(n_reviews) OVER (ORDER BY n_reviews DESC, app_id
                                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
             / SUM(n_reviews) OVER ()                          AS cum_share
    FROM v_games
)
SELECT MIN(CASE WHEN cum_share >= 0.8 THEN rn END) / MAX(n)    AS share_games_for_80pct,
       SUM(CASE WHEN rn <= 0.2 * n THEN n_reviews END) / SUM(n_reviews) AS reviews_share_top20pct
FROM ranked
""").first()
print(f"Grain jeu : {concentration['share_games_for_80pct']:.1%} des jeux concentrent 80 % des avis ; "
      f"le top 20 % des jeux en concentre {concentration['reviews_share_top20pct']:.1%}.")

26/11/18 09:55:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 09:55:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 09:55:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 09:55:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 09:55:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 09:55:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 0

+---------------------+-------+-------------+-------------------+---------------------+-----------------------+--------------------+-------------------------------+------------+---------+---------------+-------------+----------------+------------+
|primary_genre        |n_games|total_reviews|recommendation_rate|mean_reviews_per_game|median_reviews_per_game|iqr_reviews_per_game|median_game_recommendation_rate|median_price|iqr_price|rank_by_reviews|share_reviews|pareto_cum_share|pareto_class|
+---------------------+-------+-------------+-------------------+---------------------+-----------------------+--------------------+-------------------------------+------------+---------+---------------+-------------+----------------+------------+
|Action               |2291   |1725385      |0.8057             |753.1                |43                     |267                 |0.7952                         |4.99        |6.9      |1              |0.4584       |0.4584          |A           |
|Indie  

5 genre(s) sur 20 (classe A) concentrent ~80 % des avis.


26/11/18 09:56:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 09:56:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 09:56:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 09:56:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 09:56:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 09:56:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 0

Grain jeu : 8.6% des jeux concentrent 80 % des avis ; le top 20 % des jeux en concentre 91.5%.


#### Analyse Gold #1 `genre_pareto` : concentration du marché et satisfaction par genre

- **Concentration (Pareto)** : **5 genres sur 20 (classe A : Action, Indie, Strategy, Adventure, RPG) concentrent 86,2 % des avis**, et Action à lui seul en porte 45,8 %. Au grain jeu, la concentration est encore plus forte que la règle des 80/20 : **8,6 % des jeux concentrent 80 % des avis**, et le top 20 % des jeux en concentre 91,5 %. Le volume d'avis (donc l'attention des joueurs) est un marché de blockbusters.
- **Moyenne contre médiane par genre** : dans Action, un jeu reçoit en moyenne 753 avis, mais la médiane n'est que de 43 (IQR = 267). La moyenne d'un genre décrit ses quelques hits, pas ses jeux typiques. La table expose donc la médiane et l'IQR à côté de la moyenne.
- **Satisfaction** (`recommendation_rate` = part d'avis positifs, pondérée par les avis) : les mieux notés parmi les genres à fort volume sont **Adventure (87,5 %)**, Indie (86,4 %) et RPG (85,7 %). Action et Strategy sont en retrait (80,6 %), et **Massively Multiplayer est dernier (72,6 %)**, avec un prix médian de 0 £ : des *free-to-play* très joués mais plus critiqués. Ces écarts de quelques points portent sur des centaines de milliers d'avis. Le Jour 4 testera s'ils sont statistiquement significatifs (Kruskal-Wallis au grain jeu).
- **Pondération** : le taux pondéré par les avis (`recommendation_rate`) est presque toujours **supérieur** au taux médian d'un jeu du genre (`median_game_recommendation_rate`), par exemple Simulation 84,0 % contre 69,7 %. Les jeux les plus joués sont aussi les mieux notés. Les deux chiffres répondent à des questions différentes : « que pensent les joueurs du genre ? » contre « que vaut un jeu typique du genre ? ».
- **Qualité de la dimension** : `primary_genre` est le 1er tag SteamSpy *voté par les joueurs* qui est aussi un genre officiel (96,4 % des jeux, sinon repli sur le 1er genre officiel, cf. J2). Le champ `genres` étant trié par ordre alphabétique, prendre son 1er élément aurait attribué 65 % des avis à « Action » (ce biais a été corrigé lors de cet audit).
- `coalesce(1)` : la table Gold fait 20 lignes, donc **1 seul fichier Delta**. Le dashboard la lit en une requête, sans *small files*.

## 3. Table Gold #2 : Dynamique Temporelle & Croissance avec `LAG()`

### Exercice 3.1 — Série temporelle et taux d'évolution période-sur-période
Agrégez votre table `v_silver` selon son axe temporel (jour, semaine, mois ou année selon votre sujet) et utilisez la fonction de fenêtrage **`LAG(metric, 1) OVER (ORDER BY periode)`** (ou une moyenne glissante `AVG(...) OVER (ORDER BY periode ROWS BETWEEN ... PRECEDING AND CURRENT ROW)`) pour mesurer la dynamique d'évolution.

Sauvegardez le résultat dans `/lakehouse/gold/<nom_gold_2>`.

In [4]:
GOLD_TIME_PATH = "/lakehouse/gold/release_year_trends"

# Gold #2 : série annuelle par année de sortie (seul axe temporel du dataset, issu du catalogue en J2).
# Calendrier complet (SEQUENCE) pour que LAG compare bien deux années consécutives, même sans sortie.
df_gold_2 = spark.sql("""
WITH yearly AS (
    SELECT release_year,
           COUNT(*)                                                    AS n_games,
           SUM(n_reviews)                                              AS total_reviews,
           ROUND(SUM(n_reviews * recommendation_rate) / SUM(n_reviews), 4) AS recommendation_rate,
           ROUND(AVG(price), 2)                                        AS avg_price,
           PERCENTILE_APPROX(price, 0.5)                               AS median_price,
           PERCENTILE_APPROX(n_reviews, 0.5)                           AS median_reviews_per_game,
           MAX(release_date)                                           AS last_release_date
    FROM v_games
    WHERE release_year IS NOT NULL
    GROUP BY release_year
),
catalog AS (
    SELECT release_year, COUNT(*) AS catalog_games FROM v_catalog GROUP BY release_year
),
calendar AS (
    SELECT EXPLODE(SEQUENCE(MIN(release_year), MAX(release_year))) AS release_year FROM yearly
),
filled AS (
    SELECT c.release_year,
           COALESCE(y.n_games, 0)       AS n_games,
           COALESCE(k.catalog_games, 0) AS catalog_games,
           COALESCE(y.total_reviews, 0) AS total_reviews,
           y.recommendation_rate, y.avg_price, y.median_price, y.median_reviews_per_game, y.last_release_date
    FROM calendar c
    LEFT JOIN yearly y ON c.release_year = y.release_year
    LEFT JOIN catalog k ON c.release_year = k.release_year
)
SELECT release_year, n_games, catalog_games,
       ROUND(try_divide(n_games, catalog_games), 4)                            AS review_coverage,
       total_reviews, recommendation_rate, avg_price, median_price, median_reviews_per_game,
       LAG(total_reviews, 1) OVER w                                            AS prev_year_reviews,
       ROUND(try_divide(total_reviews - LAG(total_reviews, 1) OVER w,
                        LAG(total_reviews, 1) OVER w) * 100, 1)               AS reviews_yoy_pct,
       ROUND((recommendation_rate - LAG(recommendation_rate, 1) OVER w) * 100, 2) AS recommendation_delta_pts,
       ROUND(AVG(recommendation_rate) OVER (ORDER BY release_year
                                            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 4) AS recommendation_rate_ma3,
       ROUND(avg_price - LAG(avg_price, 1) OVER w, 2)                          AS avg_price_delta,
       n_games < 30                                                            AS low_sample,
       release_year = MAX(release_year) OVER ()
         AND MONTH(MAX(last_release_date) OVER ()) < 12                        AS is_partial_year,
       n_games >= 30 AND try_divide(n_games, catalog_games) >= 0.5             AS is_reliable
FROM filled
WINDOW w AS (ORDER BY release_year)
ORDER BY release_year
""")
assert df_gold_2 is not None, "TODO : Écrivez la requête SQL temporelle avec LAG() dans `df_gold_2`."

(
    df_gold_2.coalesce(1)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_TIME_PATH)
)
df_gold_2.show(20, truncate=False)

gold_2 = spark.read.format("delta").load(GOLD_TIME_PATH)
years = [r["release_year"] for r in gold_2.orderBy("release_year").select("release_year").collect()]
assert years == list(range(years[0], years[-1] + 1)), "Série annuelle continue attendue (sans trou)"
assert gold_2.agg({"total_reviews": "sum"}).first()[0] == spark.sql("SELECT SUM(n_reviews) FROM v_games").first()[0]
print("Fin de série - couverture des avis vs catalogue (is_reliable : n >= 30 et couverture >= 50 %) :")
gold_2.filter("release_year >= 2012").select(
    "release_year", "n_games", "catalog_games", "review_coverage", "total_reviews", "reviews_yoy_pct",
    "recommendation_rate", "recommendation_delta_pts", "recommendation_rate_ma3", "avg_price", "is_reliable",
).show(10, truncate=False)
reliable = gold_2.filter("is_reliable")
assert reliable.count() >= 5, "Pas assez d'années fiables pour une analyse de tendance"

26/11/18 09:56:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 09:56:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 09:56:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 09:56:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 09:56:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 09:56:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 0

+------------+-------+-------------+---------------+-------------+-------------------+---------+------------+-----------------------+-----------------+---------------+------------------------+-----------------------+---------------+----------+---------------+-----------+
|release_year|n_games|catalog_games|review_coverage|total_reviews|recommendation_rate|avg_price|median_price|median_reviews_per_game|prev_year_reviews|reviews_yoy_pct|recommendation_delta_pts|recommendation_rate_ma3|avg_price_delta|low_sample|is_partial_year|is_reliable|
+------------+-------+-------------+---------------+-------------+-------------------+---------+------------+-----------------------+-----------------+---------------+------------------------+-----------------------+---------------+----------+---------------+-----------+
|1997        |1      |1            |1.0            |78           |0.9359             |5.99     |5.99        |78                     |NULL             |NULL           |NULL             

26/11/18 09:56:46 ERROR Utils: Uncaught exception in thread executor-heartbeater
java.lang.NullPointerException: Cannot invoke "java.util.concurrent.ConcurrentHashMap.isEmpty()" because the return value of "org.apache.spark.sql.util.PartitionKeyedAccumulator.byPartition()" is null
	at org.apache.spark.sql.util.PartitionKeyedAccumulator.isZero(PartitionKeyedAccumulator.scala:51)
	at org.apache.spark.executor.Executor.$anonfun$reportHeartBeat$2(Executor.scala:1536)
	at org.apache.spark.executor.Executor.$anonfun$reportHeartBeat$2$adapted(Executor.scala:1536)
	at scala.collection.immutable.Vector.filterImpl(Vector.scala:166)
	at scala.collection.immutable.Vector.filterImpl(Vector.scala:116)
	at scala.collection.StrictOptimizedIterableOps.filterNot(StrictOptimizedIterableOps.scala:220)
	at scala.collection.StrictOptimizedIterableOps.filterNot$(StrictOptimizedIterableOps.scala:220)
	at scala.collection.immutable.Vector.filterNot(Vector.scala:116)
	at org.apache.spark.executor.Executor.$anon

Fin de série - couverture des avis vs catalogue (is_reliable : n >= 30 et couverture >= 50 %) :


+------------+-------+-------------+---------------+-------------+---------------+-------------------+------------------------+-----------------------+---------+-----------+
|release_year|n_games|catalog_games|review_coverage|total_reviews|reviews_yoy_pct|recommendation_rate|recommendation_delta_pts|recommendation_rate_ma3|avg_price|is_reliable|
+------------+-------+-------------+---------------+-------------+---------------+-------------------+------------------------+-----------------------+---------+-----------+
|2012        |304    |320          |0.95           |326573       |38.5           |0.8755             |-0.68                   |0.8838                 |8.7      |true       |
|2013        |395    |418          |0.945          |573742       |75.7           |0.8392             |-3.63                   |0.8657                 |8.7      |true       |
|2014        |1486   |1555         |0.9556         |577613       |0.7            |0.7911             |-4.81                   |0.8

#### Analyse Gold #2 `release_year_trends` : dynamique par année de sortie

- **Lecture de l'axe** : les avis n'ont pas d'horodatage. `release_year` est l'année de sortie du jeu (catalogue, jointure J2) : on compare donc des **cohortes de jeux**, pas l'évolution des avis dans le temps.
- **Couverture du dataset d'avis** : la colonne `review_coverage` (jeux avec avis / jeux du catalogue, via la dimension Silver `steam_games`) révèle un **biais de collecte**. Elle reste ≥ 91 % jusqu'en 2015 et vaut 69 % en 2016, puis **chute à 3,1 % en 2017, 1,1 % en 2018 et 1,1 % en 2019**, alors que le catalogue compte 6 357 et 8 160 sorties en 2017 et 2018. Le corpus Kaggle d'avis a été constitué avant la plupart de ces sorties. La « baisse de -76,6 % des avis » de 2017 calculée par `LAG` est donc un **artefact de collecte**, pas un effondrement du marché. L'indicateur `is_reliable` (≥ 30 jeux et couverture ≥ 50 %) retient **2006-2016** pour l'analyse et le dashboard, et écarte aussi 1997-2005, où il y a moins de 10 jeux par an.
- **Tendances sur la période fiable (2006-2016)** :
  - **Explosion de l'offre** : 395 jeux en 2013, **1 486 en 2014** (×3,8), 3 022 en 2016. C'est l'ouverture de Steam aux indépendants (*Greenlight*, 2012).
  - **Dilution de l'attention** : la médiane d'avis par jeu passe de 251 (2013) à **17 (2016)**. Plus de jeux, chacun moins visible.
  - **Satisfaction en baisse** : la moyenne glissante sur 3 ans du taux de recommandation (`recommendation_rate_ma3`) passe de **88,4 % (2012) à 79,9 % (2016)**, soit -8,5 points. Sur 2012-2016, le `LAG` montre les plus fortes baisses en 2013 (-3,6 pts) et 2014 (-4,8 pts), au moment de l'afflux de nouveaux titres.
  - **Prix moyen en baisse** : de 8,70 £ (2012-2013) à 6,12 £ (2016), dans le même sens que la multiplication des petits jeux à bas prix.
- **Prudence d'interprétation** : les cohortes anciennes encore évaluées sont surtout des classiques qui ont « survécu » (*biais du survivant*), ce qui gonfle mécaniquement leur satisfaction. La baisse observée mélange donc une vraie dégradation moyenne et un effet de sélection.
- **Technique** : le calendrier complet (`SEQUENCE` + `LEFT JOIN`) garantit que `LAG(…, 1)` compare deux années consécutives (2002 apparaît avec 0 jeu au lieu de disparaître). `try_divide` évite l'erreur ANSI sur une division par 0. L'avertissement Spark `No Partition Defined for Window operation` est attendu : une fenêtre sans `PARTITION BY` regroupe toutes les lignes sur une seule tâche. C'est sans conséquence sur 23 lignes annuelles, mais ce serait un goulot d'étranglement sur une table de faits.

## 4. Table Gold #3 (Axe complémentaire libre) & Compaction `OPTIMIZE`

### Exercice 4.1 — Produire un 3e axe d'analyse (`ROW_NUMBER() OVER (PARTITION BY ...)`) et compacter Silver
1. Créez une 3e table Gold répondant à une question spécifique de votre problématique (ex. top sous-catégorie par région/groupe via `ROW_NUMBER() OVER (PARTITION BY groupe ORDER BY ... DESC) = 1`).
2. Exécutez `OPTIMIZE delta.'{SILVER_MAIN_PATH}'` et expliquez pourquoi la multiplication de micro-fichiers pénalise le NameNode HDFS.

In [ ]:
GOLD_TOP_PATH = "/lakehouse/gold/top_games_by_genre"
MIN_REVIEWS = 50  # seuil d'éligibilité : un taux sur 3 avis n'est pas comparable à un taux sur 30 000
TOP_N = 10

# Gold #3 : Top 10 des jeux les mieux recommandés de chaque genre (ROW_NUMBER partitionné par genre).
# Tri sur la borne basse de l'intervalle de Wilson à 95 % du taux de recommandation (z = 1,96) : elle
# pénalise les petits effectifs, contrairement au taux brut (100 % sur 50 avis < 97 % sur 50 000 avis).
df_gold_3 = spark.sql(f"""
WITH scored AS (
    SELECT *,
           (recommendation_rate + 1.9208 / n_reviews
            - 1.96 * SQRT((recommendation_rate * (1 - recommendation_rate) + 0.9604 / n_reviews) / n_reviews))
           / (1 + 3.8416 / n_reviews)                                         AS wilson_lower_bound
    FROM v_games
    WHERE n_reviews >= {MIN_REVIEWS}
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY primary_genre
                              ORDER BY wilson_lower_bound DESC, n_reviews DESC, app_id) AS rank_in_genre,
           ROW_NUMBER() OVER (PARTITION BY primary_genre
                              ORDER BY n_reviews DESC, app_id)                     AS volume_rank_in_genre,
           COUNT(*) OVER (PARTITION BY primary_genre)                              AS genre_eligible_games
    FROM scored
)
SELECT primary_genre, rank_in_genre, app_id, game_name, publisher, release_year, price, price_tier,
       n_reviews, ROUND(recommendation_rate, 4) AS recommendation_rate,
       ROUND(wilson_lower_bound, 4) AS wilson_lower_bound, volume_rank_in_genre, genre_eligible_games
FROM ranked
WHERE rank_in_genre <= {TOP_N}
ORDER BY primary_genre, rank_in_genre
""")
(
    df_gold_3.coalesce(1)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_TOP_PATH)
)
gold_3 = spark.read.format("delta").load(GOLD_TOP_PATH)
gold_3.filter("rank_in_genre <= 3").select(
    "primary_genre", "rank_in_genre", "game_name", "n_reviews", "recommendation_rate", "wilson_lower_bound",
    "volume_rank_in_genre",
).show(15, truncate=False)
assert gold_3.groupBy("primary_genre", "rank_in_genre").count().filter("count > 1").count() == 0
assert gold_3.agg({"rank_in_genre": "max"}).first()[0] <= TOP_N

# --- Compaction OPTIMIZE de la table Silver ---
jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
PathHDFS = jvm.org.apache.hadoop.fs.Path


def silver_state() -> dict:
    """Fichiers référencés par la version courante (DESCRIBE DETAIL) vs fichiers physiques sur HDFS."""
    detail = spark.sql(f"DESCRIBE DETAIL delta.`{SILVER_MAIN_PATH}`").first()
    it = fs.listFiles(PathHDFS(SILVER_MAIN_PATH), True)
    physical = 0
    while it.hasNext():
        p = it.next().getPath().toString()
        physical += p.endswith(".parquet") and "/_delta_log/" not in p
    return {"version": spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_MAIN_PATH}` LIMIT 1").first()["version"],
            "active_files": detail["numFiles"], "size_mb": round(detail["sizeInBytes"] / 2**20, 1),
            "physical_parquet_files": physical}


rows_before = spark.read.format("delta").load(SILVER_MAIN_PATH).count()
before = silver_state()
print("Silver AVANT OPTIMIZE :", before)

spark.sql(f"OPTIMIZE delta.`{SILVER_MAIN_PATH}`").show(truncate=False)

after = silver_state()
print("Silver APRÈS OPTIMIZE :", after)
last_op = spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_MAIN_PATH}` LIMIT 1").first()
if last_op["version"] > before["version"] and last_op["operation"] == "OPTIMIZE":
    m = last_op["operationMetrics"]
    print(f"OPTIMIZE (version {last_op['version']}) : {m['numRemovedFiles']} fichiers compactés -> "
          f"{m['numAddedFiles']} fichier(s)")
else:
    print("Table déjà compactée : rien à regrouper, aucune nouvelle version Delta (OPTIMIZE idempotent)")
assert after["active_files"] <= before["active_files"], "OPTIMIZE ne doit jamais augmenter le nombre de fichiers actifs"
assert spark.read.format("delta").load(SILVER_MAIN_PATH).count() == rows_before, "OPTIMIZE ne modifie pas les données"
spark.catalog.clearCache()  # libère le cache de v_games (driver 2g partagé avec les autres notebooks)

+---------------------+-------------+----------------------------------------------+---------+-------------------+------------------+--------------------+
|primary_genre        |rank_in_genre|game_name                                     |n_reviews|recommendation_rate|wilson_lower_bound|volume_rank_in_genre|
+---------------------+-------------+----------------------------------------------+---------+-------------------+------------------+--------------------+
|Action               |1            |Half-Life 2                                   |4052     |0.9998             |0.9986            |100                 |
|Action               |2            |Star Wars: Battlefront 2 (Classic, 2005)      |1858     |1.0                |0.9979            |196                 |
|Action               |3            |Batman: Arkham City - Game of the Year Edition|563      |1.0                |0.9932            |400                 |
|Adventure            |1            |TY the Tasmanian Tiger           

Silver AVANT OPTIMIZE : {'version': 10, 'active_files': 15, 'size_mb': 17.8, 'physical_parquet_files': 67}


+------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|path                                                        |metrics                                                                                                                                                                                          |
+------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|hdfs://namenode:9000/lakehouse/silver/steam_reviews_enriched|{1, 15, {18243233, 18243233, 1.8243233E7, 1, 18243233}, {327663, 2676714, 1243380.5333333334, 15, 18650708}, 1, NULL, NULL, 1, 1, 15, 0, false, 0, 0, 1794995875335, 0,

Silver APRÈS OPTIMIZE : {'version': 11, 'active_files': 1, 'size_mb': 17.4, 'physical_parquet_files': 68}
OPTIMIZE (version 11) : 15 fichiers compactés -> 1 fichier(s)


26/11/18 09:58:43 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 126862 ms exceeds timeout 120000 ms
26/11/18 09:58:43 WARN SparkContext: Killing executors is not supported by current scheduler.


#### Analyse Gold #3 `top_games_by_genre` et compaction `OPTIMIZE`

**Gold #3 : les jeux les mieux recommandés de chaque genre.** `ROW_NUMBER() OVER (PARTITION BY primary_genre ORDER BY wilson_lower_bound DESC)` numérote les jeux **à l'intérieur de chaque genre** : la fenêtre est partitionnée, donc chaque genre est classé indépendamment et en parallèle. Le filtre `rank_in_genre <= 10` donne 133 lignes (10 jeux pour 11 genres, moins pour les genres plus petits ; 18 genres ont au moins un jeu éligible).
- **Pourquoi la borne de Wilson ?** Trier sur le taux brut placerait en tête des jeux à 100 % sur une poignée d'avis. La borne basse de l'intervalle de confiance de Wilson à 95 % pénalise l'incertitude des petits effectifs. Ainsi *Half-Life 2* (99,98 % sur 4 052 avis, borne 0,9986) passe devant n'importe quel jeu à 100 % sur 60 avis (borne ≈ 0,94). S'y ajoute un seuil d'éligibilité de 50 avis.
- **Qualité ≠ popularité** : pour 7 des 18 genres, le jeu classé n°1 n'est même pas dans le top 10 du genre en volume d'avis (`volume_rank_in_genre`). Exemple : Half-Life 2 est 1er en qualité mais 100e en volume dans Action. Le dashboard pourra opposer ces deux classements.
- `ROW_NUMBER` plutôt que `DENSE_RANK` : on veut **exactement** N lignes par genre, avec un ordre de départage déterministe (`n_reviews`, puis `app_id`) ; l'`assert` vérifie qu'il n'y a pas de doublon de rang.

**Compaction `OPTIMIZE` de Silver** *(mesures du 2026-11-18)*

| | Avant | Après |
|---|---|---|
| Fichiers actifs (version courante) | **15** (taille moyenne 1,2 Mo, de 0,31 à 2,55 Mo) | **1** (17,4 Mo) |
| Volume de données | 17,8 Mo | 17,4 Mo (meilleure compression d'un seul fichier) |
| Fichiers Parquet physiques sur HDFS | 33 | 34 |

- **D'où viennent les petits fichiers ?** Le chargement initial de J2 a écrit **8 fichiers** (1 par tâche Spark), puis le `MERGE` en *copy-on-write* en a réécrit **8 de plus** pour un seul fichier remplacé. Résultat : 15 fichiers actifs d'environ 1,2 Mo pour une table de 17,8 Mo, alors qu'un bloc HDFS fait 16 Mo. Chaque nouveau lot `MERGE` aggraverait la situation.
- **Pourquoi est-ce pénalisant ? (*Small Files Problem*)**
  - **NameNode** : il garde **en RAM** chaque fichier, répertoire et bloc (~150 octets par objet), quelle que soit sa taille. 15 fichiers de 1,2 Mo = 15 *inodes* + 15 blocs **partiellement remplis**, contre 1 *inode* + 2 blocs pour le même volume compacté (16 Mo + 1,4 Mo). À l'échelle d'un *data lake* (millions de micro-fichiers), la mémoire du NameNode devient la limite du cluster, ainsi que la charge des *block reports* envoyés par les DataNodes.
  - **Spark** : au moins 1 tâche par fichier, un coût fixe d'ouverture (`spark.sql.files.openCostInBytes` = 4 Mo), la lecture du *footer* Parquet de chaque fichier et un aller-retour NameNode par fichier. Sur des fichiers minuscules, on passe plus de temps à planifier et à ouvrir qu'à lire.
  - **Delta** : chaque fichier ajoute une action `add` et ses statistiques au journal, ce qui alourdit le *snapshot* à reconstruire à chaque lecture.
- **Ce que fait `OPTIMIZE`** : un *bin-packing* qui réécrit les petits fichiers en fichiers proches de `maxFileSize` (1 Go par défaut). Ici toute la table tient dans **1 fichier** et en un seul commit (`numRemovedFiles = 15`, `numAddedFiles = 1`) marqué `dataChange = false` : le nombre de lignes est identique (vérifié par `assert`) et les lecteurs en cours ne sont pas perturbés (isolation ACID).
- **Nuance importante** : le nombre de fichiers **physiques** augmente (33 → 34). OPTIMIZE ne supprime rien : les 15 anciens fichiers deviennent des actions `remove` (pierres tombales logiques) mais restent sur HDFS pour le *Time Travel* (`versionAsOf 0` de J2 fonctionne toujours). La mémoire du NameNode n'est réellement libérée qu'après un `VACUUM`, qui supprime les fichiers non référencés **plus vieux que la rétention** (7 jours par défaut). Il ne doit pas être lancé avec une rétention plus courte tant qu'on veut garder l'historique.
- **Prévention** plutôt que correction : `coalesce(1)` sur les petites tables Gold (1 fichier chacune), `optimizeWrite`/`autoCompact` de Delta à l'écriture, et `OPTIMIZE` planifié après les lots `MERGE`. Relancer cette cellule sur une table déjà compactée ne fait rien (« rien à regrouper ») : l'opération est idempotente.